In [25]:
import torch
from torch import nn
import numpy as np
from string import Template
import math

import os 
import sys
sys.path.append(os.path.abspath("./"))
from input_image_gen import generate_input_file
from conv_layer_generator import *
from dense_layer_generator import *
from codebooks_defs_generator import *

In [26]:
def conv_out_size(in_size, stride, padding, k_size):
    return int(((in_size - k_size + (2 * padding)) / stride) + 1) 

def max_pool_out_size(in_size, pool_size, stride):
    return int(((in_size - pool_size) / (stride)) + 1)

In [27]:

TILE_L2_SIZE = 2
TILE_L1_SIZE = 15

CODEBOOK_SIZE = 4
SVE_LANES = 4

N_LEARNERS = 4

In [28]:

USE_F16 = False

SAME_SEQ = True

USE_BIAS = False

USE_CODEBOOKS = True


OUTPUT_SIZE = 200   # TinyImageNet



IN_CHANNELS = 3
IN_HEIGHT = 64
IN_WIDTH = IN_HEIGHT


####################
##  VGG16   ##
####################

conv_0 = {"type": "conv",
          "in_ch": IN_CHANNELS, 
          "out_ch": 64,
          "k_size": 3, 
          "stride": 1, 
          "padding": 1}

conv_1 = {"type": "conv",
          "in_ch": conv_0["out_ch"], 
          "out_ch": 64,
          "k_size": 3,
          "stride": 1,
          "padding": 1}

max_pool_2 = {"type": "maxpool",
              "size": 2,
              "stride": 2}

###########

conv_3 = {"type": "conv",
          "in_ch": conv_1["out_ch"], 
          "out_ch": 128,
          "k_size": 3,
          "stride": 1,
          "padding": 1}

conv_4 = {"type": "conv",
          "in_ch": conv_3["out_ch"], 
          "out_ch": 128,
          "k_size": 3,
          "stride": 1,
          "padding": 1}

max_pool_5 = {"type": "maxpool",
              "size": 2,
              "stride": 2}

###########

conv_6 = {"type": "conv",
          "in_ch": conv_4["out_ch"], 
          "out_ch": 256,
          "k_size": 3,
          "stride": 1,
          "padding": 1}

conv_7 = {"type": "conv",
          "in_ch": conv_6["out_ch"], 
          "out_ch": 256,
          "k_size": 3,
          "stride": 1,
          "padding": 1}

conv_8 = {"type": "conv",
          "in_ch": conv_7["out_ch"], 
          "out_ch": 256,
          "k_size": 3,
          "stride": 1,
          "padding": 1}

max_pool_9 = {"type": "maxpool",
              "size": 2,
              "stride": 2}

###########

conv_10 = {"type": "conv",
          "in_ch": conv_8["out_ch"], 
          "out_ch": 512,
          "k_size": 3,
          "stride": 1,
          "padding": 1}

conv_11 = {"type": "conv",
          "in_ch": conv_10["out_ch"], 
          "out_ch": 512,
          "k_size": 3,
          "stride": 1,
          "padding": 1}

conv_12 = {"type": "conv",
          "in_ch": conv_11["out_ch"], 
          "out_ch": 512,
          "k_size": 3,
          "stride": 1,
          "padding": 1}

max_pool_13 = {"type": "maxpool",
              "size": 2,
              "stride": 2}

###########

conv_14 = {"type": "conv",
          "in_ch": conv_12["out_ch"], 
          "out_ch": 512,
          "k_size": 3,
          "stride": 1,
          "padding": 1}

conv_15 = {"type": "conv",
          "in_ch": conv_14["out_ch"], 
          "out_ch": 512,
          "k_size": 3,
          "stride": 1,
          "padding": 1}

conv_16 = {"type": "conv",
          "in_ch": conv_15["out_ch"], 
          "out_ch": 512,
          "k_size": 3,
          "stride": 1,
          "padding": 1}

max_pool_17 = {"type": "maxpool",
              "size": 2,
              "stride": 2}

###########

glob_avg_pool_18 = {"type": "glob_avg_pool"}

# dense_18 = {"type": "dense",
#            "out_size": 4096}

# dense_19 = {"type": "dense",
#            "out_size": 4096}

dense_19 = {"type": "dense",
           "out_size": OUTPUT_SIZE}


NN_structure = [conv_0, conv_1, max_pool_2, 
                conv_3, conv_4, max_pool_5, 
                conv_6, conv_7, conv_8, max_pool_9, 
                conv_10, conv_11, conv_12, max_pool_13, 
                conv_14, conv_15, conv_16, max_pool_17, 
                glob_avg_pool_18,
                dense_19]

In [29]:
# OUT_FOLDER = "./generated_headers/"
OUT_FOLDER = "./../VGG16_definitions/"

generate_cb_definitions(OUT_FOLDER + "codebooks_def.h", N_LEARNERS, CODEBOOK_SIZE, SVE_LANES, USE_BIAS, USE_F16, SAME_SEQ, USE_CODEBOOKS)
input_values = generate_input_file(OUT_FOLDER + "input_image.h", IN_CHANNELS, IN_HEIGHT, IN_WIDTH, USE_F16)

# These are in case the layer is a conv or max pool
input_ch = IN_CHANNELS
input_height = IN_HEIGHT
input_width = IN_WIDTH

# This is in case the layer is a dense layer
in_size = IN_CHANNELS * IN_HEIGHT * IN_WIDTH


# This list is to save all the kernel values, so to test with torch
kernels = []

# This list is to save all the dense layer values, so   to test with torch
dense_weights = []

# Final torch network, one per learner
network = [nn.ModuleDict({}) for _ in range(N_LEARNERS)]

for lay_cnt, layer in enumerate(NN_structure):
    print("[{}] {}".format(lay_cnt, layer["type"]))
    print("\t", layer)

    if layer['type'] == "conv":
        in_shape = (layer["in_ch"], input_height, input_width)
        out_channels = layer["out_ch"]
        out_height = conv_out_size(input_height, layer["stride"], layer["padding"], layer["k_size"])
        out_width = conv_out_size(input_width, layer["stride"], layer["padding"], layer["k_size"])
        out_shape = (out_channels, out_height, out_width)

        # Get the codebooks values for the learners and generate the header
        cb_values, k_values, biases_values = generate_kernel_header_file(SAME_SEQ, OUT_FOLDER + "conv_header_{}.h".format(lay_cnt), lay_cnt, N_LEARNERS, CODEBOOK_SIZE, out_channels, layer["in_ch"], layer["k_size"], layer['stride'], layer["padding"], TILE_L2_SIZE, TILE_L1_SIZE, USE_F16, USE_CODEBOOKS)
        kernels.append(k_values)


    
        # Per each learner network, force the kernel values and append the layer to the learner network
        for learner in range(N_LEARNERS):

            new_conv = nn.Conv2d(in_channels=layer['in_ch'],
                                                        out_channels=layer["out_ch"],
                                                        kernel_size=(layer["k_size"], layer["k_size"]),
                                                        stride = layer['stride'],
                                                        padding=layer["padding"],
                                                        bias = USE_BIAS)
            with torch.no_grad():
                new_conv.weight.copy_(torch.tensor(k_values[learner]).view(layer["out_ch"], layer['in_ch'], layer["k_size"], layer["k_size"]))

                if USE_BIAS:
                    new_conv.bias.copy_(torch.tensor(biases_values[learner]))

            network[learner]["conv{}".format(lay_cnt)] = new_conv


        input_ch = out_channels
        input_height = out_height
        input_width = out_width

        in_size = out_channels * out_height * out_width


    elif layer["type"] == "maxpool":
        in_shape = (input_ch, input_height, input_width)
        out_channels = input_ch
        out_height = max_pool_out_size(input_height, layer["size"], layer["stride"])
        out_width = max_pool_out_size(input_width, layer["size"], layer["stride"])
        out_shape = (out_channels, out_height, out_width)

        # Per each learner network, append the layer to the network
        for learner in range(N_LEARNERS):
            new_maxpool = nn.MaxPool2d(layer["size"], stride=layer["stride"])
            # new_maxpool = nn.MaxPool2d(layer["size"])
            network[learner]["maxpool{}".format(lay_cnt)] = new_maxpool

        input_ch = out_channels
        input_height = out_height
        input_width = out_width

        in_size = out_channels * out_height * out_width

    elif layer['type'] == "glob_avg_pool":
        in_shape = (input_ch, input_height, input_width)
        out_channels = input_ch
        out_height = 1
        out_width = 1
        out_shape = (out_channels, out_height, out_width)
        
        for learner in range(N_LEARNERS):
            new_glob_avgPool = nn.AdaptiveAvgPool2d((1, 1))
            network[learner]["glob_avgPool{}".format(lay_cnt)] = new_glob_avgPool
        
        input_ch = out_channels
        input_height = out_height
        input_width = out_width

        in_size = out_channels * out_height * out_width

    elif layer["type"] == "dense":
        print("DENSE: ", in_size)
        in_shape = in_size
        out_size = layer["out_size"]

        dense_values, biases_values = generate_template_dense(SAME_SEQ, OUT_FOLDER + "dense_header_{}.h".format(lay_cnt), lay_cnt, N_LEARNERS, CODEBOOK_SIZE, 200, in_size, out_size, USE_F16, USE_CODEBOOKS)
        dense_weights.append(dense_values)

        for learner in range(N_LEARNERS):

            new_dense = nn.Linear(in_shape, layer["out_size"], bias=USE_BIAS)
            
            with torch.no_grad():
                new_dense.weight.copy_(torch.Tensor(dense_values[learner]).view(out_size, in_shape))

                if USE_BIAS:
                    new_dense.bias.copy_(torch.tensor(biases_values[learner]))

            network[learner]["dense{}".format(lay_cnt)] = new_dense

        out_shape = out_size

        in_size = out_shape
        
    else:
        print("ERROR!")
        exit(1)



    print("In shape:", in_shape)
    print("Out shape:", out_shape)
    print()

[0] conv
	 {'type': 'conv', 'in_ch': 3, 'out_ch': 64, 'k_size': 3, 'stride': 1, 'padding': 1}
In shape: (3, 64, 64)
Out shape: (64, 64, 64)

[1] conv
	 {'type': 'conv', 'in_ch': 64, 'out_ch': 64, 'k_size': 3, 'stride': 1, 'padding': 1}
In shape: (64, 64, 64)
Out shape: (64, 64, 64)

[2] maxpool
	 {'type': 'maxpool', 'size': 2, 'stride': 2}
In shape: (64, 64, 64)
Out shape: (64, 32, 32)

[3] conv
	 {'type': 'conv', 'in_ch': 64, 'out_ch': 128, 'k_size': 3, 'stride': 1, 'padding': 1}
In shape: (64, 32, 32)
Out shape: (128, 32, 32)

[4] conv
	 {'type': 'conv', 'in_ch': 128, 'out_ch': 128, 'k_size': 3, 'stride': 1, 'padding': 1}
In shape: (128, 32, 32)
Out shape: (128, 32, 32)

[5] maxpool
	 {'type': 'maxpool', 'size': 2, 'stride': 2}
In shape: (128, 32, 32)
Out shape: (128, 16, 16)

[6] conv
	 {'type': 'conv', 'in_ch': 128, 'out_ch': 256, 'k_size': 3, 'stride': 1, 'padding': 1}
In shape: (128, 16, 16)
Out shape: (256, 16, 16)

[7] conv
	 {'type': 'conv', 'in_ch': 256, 'out_ch': 256, 'k_siz

In [30]:
input = torch.Tensor(input_values).view(IN_CHANNELS, IN_HEIGHT, IN_WIDTH)

print(input.shape)

for ens in range(N_LEARNERS):
    print("\n=============== LEARNER {} ===============\n".format(ens))
    # print(input)
    x = network[ens]['conv0'](input)
    print(x.shape)
    # print(x)
    # break
    x = torch.relu(x)

    x = network[ens]['conv1'](x)
    print(x.shape)
    # print(x)
    # break
    x = torch.relu(x)

    x = network[ens]['maxpool2'](x)
    print(x.shape)
    # print(x)
    # break

    print()
    ########################

    x = network[ens]['conv3'](x)
    print(x.shape)
    # print(x)
    # break
    x = torch.relu(x)

    x = network[ens]['conv4'](x)
    print(x.shape)
    # print(x)
    # break
    x = torch.relu(x)

    x = network[ens]['maxpool5'](x)
    print(x.shape)
    # print(x)
    # break

    print()
    ########################

    x = network[ens]['conv6'](x)
    print(x.shape)
    # print(x)
    # break
    x = torch.relu(x)

    x = network[ens]['conv7'](x)
    print(x.shape)
    # print(x)
    # break
    x = torch.relu(x)

    x = network[ens]['conv8'](x)
    print(x.shape)
    # print(x)
    # break
    x = torch.relu(x)

    x = network[ens]['maxpool9'](x)
    print(x.shape)
    # print(x)
    # break

    print()
    ########################

    x = network[ens]['conv10'](x)
    print(x.shape)
    # print(x)
    # break
    x = torch.relu(x)

    x = network[ens]['conv11'](x)
    print(x.shape)
    # print(x)
    # break
    x = torch.relu(x)

    x = network[ens]['conv12'](x)
    print(x.shape)
    # print(x)
    # break
    x = torch.relu(x)

    x = network[ens]['maxpool13'](x)
    print(x.shape)
    # print(x)
    # break

    print()
    ########################

    x = network[ens]['conv14'](x)
    print(x.shape)
    # print(x)
    # break
    x = torch.relu(x)

    x = network[ens]['conv15'](x)
    print(x.shape)
    # print(x)
    # break
    x = torch.relu(x)

    x = network[ens]['conv16'](x)
    print(x.shape)
    # print(x)
    # break
    x = torch.relu(x)

    x = network[ens]['maxpool17'](x)
    print(x.shape)
    # print(x)
    # break

    print()
    ########################

    x = network[ens]["glob_avgPool18"](x)
    x = x.flatten()
    print(x.shape)
    # print(x)


    x = network[ens]['dense19'](x)
    print(x.shape)
    print(x)
    # break




torch.Size([3, 64, 64])

=============== LEARNER 0 ===============

torch.Size([64, 64, 64])
torch.Size([64, 64, 64])
torch.Size([64, 32, 32])

torch.Size([128, 32, 32])
torch.Size([128, 32, 32])
torch.Size([128, 16, 16])

torch.Size([256, 16, 16])
torch.Size([256, 16, 16])
torch.Size([256, 16, 16])
torch.Size([256, 8, 8])

torch.Size([512, 8, 8])
torch.Size([512, 8, 8])
torch.Size([512, 8, 8])
torch.Size([512, 4, 4])

torch.Size([512, 4, 4])
torch.Size([512, 4, 4])
torch.Size([512, 4, 4])
torch.Size([512, 2, 2])

torch.Size([512])
torch.Size([200])
tensor([-5.0076e+35, -5.1202e+35, -4.0850e+35, -4.7023e+35, -4.6636e+35,
        -5.3969e+35, -5.0670e+35, -4.9230e+35, -5.5313e+35, -4.6342e+35,
        -5.0518e+35, -4.6335e+35, -4.7456e+35, -4.3687e+35, -5.1475e+35,
        -4.5487e+35, -4.8677e+35, -5.4591e+35, -4.4258e+35, -5.3710e+35,
        -5.3934e+35, -5.1285e+35, -4.2672e+35, -4.5954e+35, -4.3110e+35,
        -4.5191e+35, -4.9839e+35, -4.6955e+35, -5.4093e+35, -4.8684e+35,
      